# HTML and Webscraping

In [1]:
import pandas as pd

## Scraping an HTML table

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

In [2]:
# YOUR CODE HERE

import requests
headers = {"User-Agent": "GSB5544-class-experience/1.0 (Cal Poly; educational use)"} 
url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
response = requests.get(url, headers = headers)
print(response.status_code)







200


2\. Use Beautiful Soup to parse this string into a tree called `soup`

In [3]:
# YOUR CODE HERE
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content, "html.parser")
print(type(soup))
soup.title.text

<class 'bs4.BeautifulSoup'>


'List of United States cities by population - Wikipedia'

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

In [4]:
# YOUR CODE HERE
tables = soup.find_all("table")
len(tables)

10

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?



In [5]:
# YOUR CODE HERE
#<div class="shortdescription nomobile noexcerpt noprint searchaux" 
# style="display:none" about="#mwt1" typeof="mw:Transclusion" 
# for number, table in enumerate(soup.find_all("table"), start=1):
    # print(f"Table {number}")
    # print("class:", table.get("class"))
    # print("style:", table.get("style"))
    # print()

The table has these attributes:
Table 3
class: ['sortable', 'wikitable', 'sticky-header-multi', 'static-row-numbers', 'sort-under', 'col1left', 'col2center']
style: text-align:right

5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

In [6]:
# YOUR CODE HERE
target_class = (
    "sortable wikitable sticky-header-multi static-row-numbers "
    "sort-under col1left col2center"
)

tables = [
    table for table in soup.find_all("table")
    if " ".join(table.get("class", [])) == target_class
    and table.get("style") == "text-align:right"
]

print(len(tables))

1


6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [7]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

In [15]:
# YOUR CODE HERE
city = table.find_all("tr")[3]



8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

In [54]:
# YOUR CODE HERE

# Find the first table cell, extract its text, and remove extra whitespace
# Extract the city name and remove the citation marker
name = city.find_all("td")[0].get_text(strip=True).replace("[c]", "")
name

'New York'

9\. Extract the city's state and store it as `state`.

In [55]:
# YOUR CODE HERE
state = city.find_all("td")[1].get_text(strip=True)
state

'NY'

10\. Extract the city's population and store is as `population`.

In [56]:
# YOUR CODE HERE
popn = city.find_all("td")[2].get_text(strip=True)
popn

'8,584,629'

11\. Extract the city's area and store is as `area`.

In [57]:
# YOUR CODE HERE
area = city.find_all("td")[3].get_text(strip=True)
area

'8,804,190'

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


In [58]:
# YOUR CODE HERE. ADD AS MANY CELLS AS NEEDED
import time
# initialize an empty list
rows = []

# iterate over all rows in the faculty table
for city in table.find_all("tr")[3:]:

    name = city.find_all("td")[0].get_text(strip=True).replace("[c]", "")
    state = city.find_all("td")[1].get_text(strip=True)
    popn = city.find_all("td")[2].get_text(strip=True)
    area = city.find_all("td")[3].get_text(strip=True)

# Add the city's information to rows
    rows.append([name, state, popn, area])

df = pd.DataFrame(
    rows,
    columns=["city", "state", "popn", "area"]
)

time.sleep(0.5)

13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

In [8]:
# YOUR CODE HERE
# Pandas scraping approach
from io import StringIO

pd.read_html(StringIO(response.text))[2]

Municipality   ST 2025 estimate 2020 census  Change 2020 land area  \
    Municipality   ST 2025 estimate 2020 census  Change            mi2   
0            NaN  NaN           NaN         NaN     NaN            NaN   
1    New York[c]   NY     8584629.0   8804190.0  −2.49%          300.5   
2    Los Angeles   CA     3869089.0   3898747.0  −0.76%          469.5   
3        Chicago   IL     2731585.0   2746388.0  −0.54%          227.7   
4        Houston   TX     2397315.0   2304580.0  +4.02%          640.4   
..           ...  ...           ...         ...     ...            ...   
344   San Angelo   TX      100640.0     99893.0  +0.75%           59.7   
345       Edmond   OK      100479.0     94428.0  +6.41%           84.6   
346    Davenport   IA      100358.0    101724.0  −1.34%           63.8   
347      Deltona   FL      100267.0     93692.0  +7.02%           37.3   
348     Longmont   CO      100109.0     98885.0  +1.24%           28.8   

            2020 density                                        Location  
        km2        / mi2    / km2                               Location  
0       NaN          NaN      NaN                                    NaN  
1     778.3      29298.0  11312.0    40°40′N 73°56′W﻿ / ﻿40.66°N 73.94°W  
2    1216.0       8304.0   3206.0  34°01′N 118°25′W﻿ / ﻿34.02°N 118.41°W  
3     589.7      12061.0   4657.0    41°50′N 87°41′W﻿ / ﻿41.84°N 87.68°W  
4    1658.6       3599.0   1390.0    29°47′N 95°23′W﻿ / ﻿29.79°N 95.39°W  
..      ...          ...      ...                                    ...  
344   154.6       1673.0    646.0  31°26′N 100°27′W﻿ / ﻿31.44°N 100.45°W  
345   219.1       1116.0    431.0    35°40′N 97°25′W﻿ / ﻿35.67°N 97.41°W  
346   165.2       1594.0    615.0    41°34′N 90°36′W﻿ / ﻿41.56°N 90.60°W  
347    96.6       2512.0    970.0    28°55′N 81°13′W﻿ / ﻿28.91°N 81.21°W  
348    74.6       3434.0   1326.0  40°10′N 105°06′W﻿ / ﻿40.17°N 105.10°W  

[349 rows x 10 columns]

## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

In [62]:
# YOUR CODE HERE
url = "https://www.scrapethissite.com/pages/forms/"
headers = {"User-Agent": "GSB5544-class-experience/1.0 (Cal Poly; educational use)"} 


response = requests.get(url, headers = headers)
print(response.status_code)


soup = BeautifulSoup(response.content, "html.parser")
print(type(soup))
soup.title.text


200
<class 'bs4.BeautifulSoup'>


'Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping'

2\. Find the main table on this page and store it as `table`.

In [ ]:
# YOUR CODE HERE
table = soup.find("table")

3\. Extract the information from the cells of this table into a Pandas data frame.

In [67]:

# YOUR CODE HERE

rows = []
# Get the column names from the header row
headers = [
    th.get_text(strip=True)
    for th in table.find("tr").find_all("th")
]

# iterate over all rows in the faculty table
# Loop through each hockey statistics row
for tr in table.find_all("tr")[1:]:
    cells = tr.find_all("td")
    rows.append([cell.get_text(strip=True) for cell in cells])

# Create the data frame
df = pd.DataFrame(rows, columns=headers)


4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [68]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [69]:
for link in links:
  print(link.attrs["href"])

/pages/forms/?page_num=1
/pages/forms/?page_num=2
/pages/forms/?page_num=3
/pages/forms/?page_num=4
/pages/forms/?page_num=5
/pages/forms/?page_num=6
/pages/forms/?page_num=7
/pages/forms/?page_num=8
/pages/forms/?page_num=9
/pages/forms/?page_num=10
/pages/forms/?page_num=11
/pages/forms/?page_num=12
/pages/forms/?page_num=13
/pages/forms/?page_num=14
/pages/forms/?page_num=15
/pages/forms/?page_num=16
/pages/forms/?page_num=17
/pages/forms/?page_num=18
/pages/forms/?page_num=19
/pages/forms/?page_num=20
/pages/forms/?page_num=21
/pages/forms/?page_num=22
/pages/forms/?page_num=23
/pages/forms/?page_num=24
/pages/forms/?page_num=1


Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

In [72]:
request_headers = {
    "User-Agent": "GSB5544-class-experience/1.0 (Cal Poly; educational use)"
}

column_headers = [
    th.get_text(strip=True)
    for th in table.find("tr").find_all("th")
]

all_rows = []

for link in links:
    href = link.get("href")
    link_text = link.get_text(strip=True).lower()

    if not href or link_text in ["previous", "next"]:
        continue

    page_url = urljoin(url, href)
    response = requests.get(page_url, headers=request_headers)
    page_soup = BeautifulSoup(response.content, "html.parser")
    page_table = page_soup.find("table")

    for tr in page_table.find_all("tr"):
        cells = tr.find_all("td")

        if cells:
            all_rows.append([
                cell.get_text(strip=True)
                for cell in cells
            ])

df = pd.DataFrame(all_rows, columns=column_headers)

time.sleep(0.5)